In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.image as mpimg
from matplotlib.colors import ListedColormap

## Teorie
    Simulace ekosystému lesa s trávou, králíky a vlky pomocí mřížkového modelu s animací
    Tráva může náhodně růst na prázdných políčkách, králíci jedí trávu a vlci králíky. Oba druhy se mohou pohybovat po lese a množit se.
    Následně zkoumám který druh přežije, nebo jestli systém bude vyrovnaný.

In [21]:
# --- NASTAVENÍ SIMULACE ---
N = 15 #velikost lesa
PRAZDNO, TRAVA, KRALIK, VLK = 0, 1, 2, 3 #co bude v lese

les = np.random.choice([PRAZDNO, TRAVA, KRALIK, VLK], size=(N, N), p=[0.87, 0.05, 0.05, 0.03]) #vytvoření lesa, jako dvourozměrného pole

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
# Načtení obrázků pro trávu, králíka a vlka
rabbit_img = mpimg.imread('/content/drive/MyDrive/UJEP/2025-26/MSW/Zapocet-jupyter/obrazky/rabbit2_small.png')
grass_img = mpimg.imread('/content/drive/MyDrive/UJEP/2025-26/MSW/Zapocet-jupyter/obrazky/grass2_small.png')
wolf_img = mpimg.imread('/content/drive/MyDrive/UJEP/2025-26/MSW/Zapocet-jupyter/obrazky/wolf2_small.png')

In [14]:
def vloz_ikonu(ax, img, x, y, scale=0.4):
    ab = AnnotationBbox(OffsetImage(img, zoom=scale), (y, x), frameon=False, pad=0) #vytvoření obrázku pro plátno
    ax.add_artist(ab) #vložení obrázku do plátna

In [15]:
def vykresli_stav(ax, les_data): #aktualizace animace
    for artist in list(ax.artists): artist.remove() #smazání všeho předchozího na plátně
    #projde celou matici
    for x in range(N): #řádky
        for y in range(N): #sloupce
            #podle toho co v buňce je, tak to vykreslí
            if les_data[x, y] == TRAVA: vloz_ikonu(ax, grass_img, x, y)
            elif les_data[x, y] == KRALIK: vloz_ikonu(ax, rabbit_img, x, y)
            elif les_data[x, y] == VLK: vloz_ikonu(ax, wolf_img, x, y)

In [26]:
fig, ax = plt.subplots(figsize=(7, 7)) #plocha pro vykreslení
ax.axis('off') #skrytí čísel, aby to nevypadalo jako graf
hnedamapa = ListedColormap(['saddlebrown']) #vytvoření barevné palety s hnědou
ax.imshow(np.full((N, N), 0.5), cmap=hnedamapa, vmin=0, vmax=1) #vytvoření matice NxN a pokrytí hnědou
title_text = ax.text(0.5, 1.05, "Ekosystém: Krok 0", transform=ax.transAxes, ha="center", fontsize=16) #titulek

def aktualizuj_les(krok):
    global les #pouze jeden les v applikaci
    novy_les = les.copy() #pro zapsání nových dat pro cyklus

    for x in range(N):
        for y in range(N):
            # 1. RŮST TRÁVY
            if les[x, y] == PRAZDNO and np.random.rand() < 0.03:
                novy_les[x, y] = TRAVA

            # 2. LOGIKA KRÁLÍKA A VLKA PRO POHYB
            elif les[x, y] in [KRALIK, VLK]:
                typ = les[x, y] #kdo se pohybuje
                mozne_pohyby = [] #možnosti kam se mohu pohnout
                #projde okolí políčka na kterém stojí
                for dx in [-1, 0, 1]: #nahoru, zůstat, dolů
                    for dy in [-1, 0, 1]: #vlevo, zůstat, vpravo
                        if dx == 0 and dy == 0: continue #tady už stojí, nepřidávám
                        nx, ny = x + dx, y + dy #souřadnice nového políčka
                        if 0 <= nx < N and 0 <= ny < N and novy_les[nx, ny] in [PRAZDNO, TRAVA]: #kontrola že se pohybuje v rozmezí lesa a že jde na políčko kde je prázdno nebo tráva
                            mozne_pohyby.append((nx, ny)) #pokud podmínky platí, přidá políčko jako možný pohyb

                if mozne_pohyby: #pokud se může hýbat, vybere si náhodné políčko z možných a za sebou nechá prázdno
                    nx, ny = mozne_pohyby[np.random.randint(len(mozne_pohyby))]
                    novy_les[x, y] = PRAZDNO
                    novy_les[nx, ny] = typ

            #Logika jídla a rozmnožování
            elif les[x, y] == KRALIK:
                muze_se_mnozit = False
                mozne_policka_rozmnozeni = []
                # Pokud v okolí 1 políčka vidí trávu, sní ji + rozmnožení
                for dx in [-1, 0, 1]: #nahoru, zůstat, dolů
                    for dy in [-1, 0, 1]: #vlevo, zůstat, vpravo
                        if dx == 0 and dy == 0: continue #tady už stojí
                        nx, ny = x + dx, y + dy #souřadnice nového políčka

                        #jídlo
                        if les[nx, ny] == TRAVA:
                            novy_les[nx, ny] = PRAZDNO # Snědl trávu

                        #množení
                        if novy_les[nx, ny] == KRALIK:
                            muze_se_mnozit = True
                        if 0 <= nx < N and 0 <= ny < N and novy_les[nx, ny] in [PRAZDNO, TRAVA]: #kontrola že se pohybuje v rozmezí lesa a že jde na políčko kde je prázdno nebo tráva
                            mozne_policka_rozmnozeni.append((nx, ny)) #pokud podmínky platí, přidá políčko jako možné pro mládě
                if muze_se_mnozit and mozne_policka_rozmnozeni: #pokud má okolo volné políčko a stejný druh, má šanci se rozmnožit
                    if np.random.randint(0, 100) <= 5: #šance na rozmnožení
                        nx, ny = mozne_policka_rozmnozeni[np.random.randint(len(mozne_policka_rozmnozeni))]
                        novy_les[nx, ny] = KRALIK

            # Pokud v okolí vidí králíka, sežere ho
            elif les[x, y] == VLK:
                muze_se_mnozit = False
                mozne_policka_rozmnozeni = []
                #pokud vidí v okolí králíka, sní ho + rozmnožení
                for dx in [-1, 0, 1]: #nahoru, zůstat, dolů
                    for dy in [-1, 0, 1]: #vlevo, zůstat, vpravo
                        if dx == 0 and dy == 0: continue #tady už stojí
                        nx, ny = x + dx, y + dy #souřadnice nového políčka

                        #jídlo
                        if les[nx, ny] == KRALIK:
                            novy_les[nx, ny] = PRAZDNO # Králík zmizel

                        #množení
                        if novy_les[nx, ny] == VLK:
                            muze_se_mnozit = True
                        if 0 <= nx < N and 0 <= ny < N and novy_les[nx, ny] in [PRAZDNO, TRAVA]: #kontrola že se pohybuje v rozmezí lesa a že jde na políčko kde je prázdno nebo tráva
                            mozne_policka_rozmnozeni.append((nx, ny)) #pokud podmínky platí, přidá políčko jako možné pro mládě
                if muze_se_mnozit and mozne_policka_rozmnozeni: #pokud má okolo volné políčko a stejný druh, má šanci se rozmnožit
                    if np.random.randint(0, 100) <= 5: #šance na rozmnožení
                        nx, ny = mozne_policka_rozmnozeni[np.random.randint(len(mozne_policka_rozmnozeni))]
                        novy_les[nx, ny] = VLK


    les = novy_les #nový stav lesa
    vykresli_stav(ax, les) #vykreslení další fáze
    title_text.set_text(f"Ekosystém: Krok {krok + 1}") #titulek


# Spuštění animace

# (nefunguje v google colab, ale ve VS code ano)
#vykresli_stav(ax, les) #vykreslení počátečního stavu 0
#ani = animation.FuncAnimation(fig, aktualizuj_les, frames=100, interval=1000) #animace, kde ze odehrává, volá funkci pro obnovení lesa, má běžet x kroků, jak často se mění snímek
#plt.show() #zobrazení

# (funguje v colab)
from IPython.display import HTML
# Vytvoření animace
ani = animation.FuncAnimation(fig, aktualizuj_les, frames=50, interval=1000, blit=False)
# Zavřeme okno, aby se animace v Colabu "neprala" s jinými procesy
plt.close()
# Zobrazení animace
HTML(ani.to_jshtml())